# GPT-2 Speech Generation — Exploration

Interactive playground for the fine-tuned GPT-2 congressional-speech model.

The `load_model` and `generate_speech` functions are defined **inline** in this
notebook (mirroring [`generate_speech.py`](generate_speech.py)) so it's fully
self-contained. It loads the checkpoint from `outputs/gpt2/final`.

**Sections**
1. Setup & paths
2. Define the model + generation functions
3. Load (warm) the model once
4. Single generation
5. Temperature sweep
6. Sampling exploration (top_p / top_k / multiple samples)
7. Free-form scratchpad

## 1. Setup & paths

Resolve the repo root by walking up from the current working directory until we find
`outputs/gpt2/final`. This makes the notebook robust whether the kernel starts in
`analysis/` or the repo root.

In [ ]:
from pathlib import Path


def find_model_dir(target=Path("outputs/gpt2/final")):
    """Walk up from CWD to locate the GPT-2 checkpoint, return its absolute path."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / target
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {target} above {Path.cwd()}. "
        "Start the kernel inside the repo."
    )


MODEL_DIR = str(find_model_dir())
print("Model dir:", MODEL_DIR)

## 2. Define the model + generation functions

`load_model` is `lru_cache`d on `(model_dir, device)`, so weights load from disk once
and every later call returns the same warm objects. `generate_speech` samples text
conditioned on a prompt via nucleus sampling. Device is auto-selected
(CUDA → Apple MPS → CPU).

In [ ]:
from functools import lru_cache

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel


def pick_device() -> torch.device:
    """Prefer CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@lru_cache(maxsize=2)
def load_model(model_dir: str = MODEL_DIR, device: str | None = None):
    """Load (and cache) the fine-tuned GPT-2 LM-head model and tokenizer.

    Cached on (model_dir, device): the first call loads weights from disk; later
    calls with the same args return the same warm objects.

    Returns (tokenizer, model, device).
    """
    dev = torch.device(device) if device else pick_device()
    print(f"Loading GPT-2 from {model_dir} onto {dev}")

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    tokenizer.pad_token = tokenizer.eos_token

    model = GPT2LMHeadModel.from_pretrained(model_dir)
    model.config.pad_token_id = tokenizer.eos_token_id
    model.eval()
    model.to(dev)

    return tokenizer, model, dev


def generate_speech(
    prompt: str,
    *,
    model_dir: str = MODEL_DIR,
    device: str | None = None,
    max_new_tokens: int = 200,
    temperature: float = 0.9,
    top_p: float = 0.95,
    top_k: int = 50,
    repetition_penalty: float = 1.2,
    no_repeat_ngram_size: int = 3,
    num_return_sequences: int = 1,
    seed: int | None = None,
) -> list[str]:
    """Generate speech text conditioned on ``prompt`` via nucleus sampling.

    Args:
        prompt: Opening text to condition on, e.g. "Mr. Speaker, I rise today".
        max_new_tokens: How many tokens to generate beyond the prompt.
        temperature: Sampling sharpness. Higher = more random/creative.
        top_p: Nucleus sampling cutoff (cumulative probability mass).
        top_k: Cap on candidate tokens per step (0 disables).
        repetition_penalty: >1.0 discourages repeating tokens (GPT-2 loops a lot).
        no_repeat_ngram_size: Forbid repeating any n-gram of this size.
        num_return_sequences: How many independent samples to return.
        seed: If set, makes generation reproducible.

    Returns:
        List of generated strings (each includes the prompt).
    """
    if not prompt or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")

    tokenizer, model, dev = load_model(model_dir, device)

    if seed is not None:
        torch.manual_seed(seed)
        if dev.type == "cuda":
            torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(dev)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )

    return [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]

## 3. Load (warm) the model once

This pays the disk-load cost once (~1.4 GB into memory). Every `generate_speech`
call below reuses these warm objects.

In [ ]:
tokenizer, model, device = load_model(MODEL_DIR)
print(f"Loaded onto {device} — {sum(p.numel() for p in model.parameters()):,} parameters")

## 4. Single generation

Edit `PROMPT` and re-run. `seed` is set for reproducibility while exploring —
remove it for fresh samples each run.

In [ ]:
PROMPT = "Mr. Speaker, I rise today"

speeches = generate_speech(
    PROMPT,
    max_new_tokens=200,
    temperature=0.9,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    seed=42,
)

print(speeches[0])

## 5. Temperature sweep

Same prompt and seed across temperatures to feel how sampling sharpness changes
the output. Lower = safer/repetitive, higher = more creative/chaotic.

In [ ]:
PROMPT = "Mr. Speaker, I rise today"

for temp in (0.5, 0.7, 0.9, 1.1):
    out = generate_speech(PROMPT, max_new_tokens=120, temperature=temp, seed=42)[0]
    print(f"\n{'=' * 70}\ntemperature = {temp}\n{'=' * 70}")
    print(out)

## 6. Sampling exploration (top_p / top_k / multiple samples)

Draw several independent samples from the same prompt to gauge variety, and try a
tighter nucleus (`top_p`) for more focused output.

In [ ]:
PROMPT = "The American people deserve"

samples = generate_speech(
    PROMPT,
    max_new_tokens=120,
    temperature=0.9,
    top_p=0.9,
    top_k=40,
    num_return_sequences=3,
)

for i, s in enumerate(samples, 1):
    print(f"\n----- sample {i}/{len(samples)} -----")
    print(s)

## 7. Free-form scratchpad

Your space — tweak prompt and any knob. See the `generate_speech` docstring above
for the full parameter list and what each does.

In [ ]:
PROMPT = ""

for s in generate_speech(
    PROMPT or "Mr. Speaker, I rise today",
    max_new_tokens=200,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.2,
):
    print(s)